# chess-gnn: RL self-play fine-tuning

Fine-tunes an SL-pretrained checkpoint with **AlphaZero-style self-play**:
each self-play move runs a short PUCT MCTS, and the visit distribution is
used as a dense policy target (cross-entropy), with the game outcome as the
value target. Root Dirichlet noise is applied during AlphaZero self-play to
broaden the visit target, while evaluation/ranking keeps MCTS deterministic.
This gives signal on every move even when games draw.

For `algo="az"`, self-play batches MCTS leaf evaluations across the active
games in each iteration and within each tree. `games_per_iter > 1` creates
cross-game batches, while `mcts_batch_size > 1` collects several pending leaves
from the same tree using virtual loss before one model call. Larger values are
most useful on GPU; `mcts_sims` still controls search quality and total work.

The alternative `algo="ppo"` path uses raw-policy self-play with a clipped-ratio
surrogate and the value head as baseline.

**Prerequisite:** an SL checkpoint (run `notebooks/train_sl.ipynb` first, or
supply an existing `sl_final.pt`). For MCTS-based fine-tuning, prefer SL
checkpoints trained with value supervision so leaf evaluations are meaningful.
Starting RL from a randomly-initialized model in chess is extremely slow and
unlikely to learn anything useful.


In [1]:
# --- environment setup ---------------------------------------------------
import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
print("colab:", IN_COLAB)

REPO_URL = ""  # optional: set to your GitHub URL to auto-clone on Colab
REPO_DIR = pathlib.Path("/content/chess") if IN_COLAB else pathlib.Path.cwd().parent

if IN_COLAB:
    if REPO_URL and not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    elif not REPO_DIR.exists():
        from google.colab import files  # type: ignore
        print("No REPO_URL set — upload a zip of the repo (containing src/chess_gnn).")
        up = files.upload()
        name = next(iter(up))
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["unzip", "-q", name, "-d", str(REPO_DIR)])

    subprocess.check_call([
        "pip", "install", "-q",
        "torch", "torch-geometric", "python-chess", "zstandard", "tqdm",
    ])
    subprocess.check_call(["pip", "install", "-q", "-e", str(REPO_DIR)])

src_path = str((REPO_DIR / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
print("torch:", torch.__version__)
if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
# DEVICE = "cpu"
print("device:", DEVICE)

colab: True
torch: 2.10.0+cu128
device: cuda


In [2]:
# --- locate the SL checkpoint to fine-tune -------------------------------
# On Colab you may want to mount Drive to persist checkpoints across sessions:
#   from google.colab import drive; drive.mount('/content/drive')
# and set SL_CKPT / RL_CKPT_DIR accordingly.

SL_CKPT = REPO_DIR / "checkpoints" / "sl" / "sl_final.pt"
RL_CKPT_DIR = REPO_DIR / "checkpoints" / "rl"
RL_CKPT_DIR.mkdir(parents=True, exist_ok=True)

assert SL_CKPT.exists(), f"SL checkpoint not found at {SL_CKPT}. Run train_sl.ipynb first or set SL_CKPT."
print("starting from:", SL_CKPT, f"({SL_CKPT.stat().st_size / 1e6:.1f} MB)")

# Read arch config from the SL checkpoint so we can rebuild the model consistently.
sl_state = torch.load(SL_CKPT, map_location="cpu")
ARCH = sl_state.get("config")
if ARCH is None:
    # Pre-config checkpoint — set these to match how it was trained.
    ARCH = {"hidden_dim": 128, "num_layers": 4, "num_heads": 4}
    print("[warn] no config in ckpt; assuming", ARCH)
else:
    print("arch:", ARCH)

starting from: /content/chess/checkpoints/sl/sl_final.pt (4.4 MB)
arch: {'hidden_dim': 128, 'num_layers': 4, 'num_heads': 4, 'dropout': 0.0}


In [3]:
# --- sanity: play one self-play game with the SL model -------------------
from chess_gnn.model import load_model
from chess_gnn.selfplay import play_self_game

probe = load_model(SL_CKPT, device=DEVICE, **ARCH)
traj = play_self_game(probe, device=DEVICE, temperature=0.7, max_plies=100)
print(f"result: {traj.result} | plies: {len(traj.positions)} | "
      f"final reward sample: {traj.rewards[-1] if traj.rewards else None}")

result: * | plies: 100 | final reward sample: -0.0


In [4]:
# --- fine-tune -----------------------------------------------------------
from chess_gnn.train_rl import train as train_rl

# AlphaZero-style fine-tune.
# Set `algo="ppo"` for the raw-policy PPO path shown in the inactive call below.
train_rl(
    sl_ckpt=SL_CKPT,
    ckpt_dir=RL_CKPT_DIR,
    iterations=50,           # bump to ~200+ for real fine-tuning
    games_per_iter=16,
    epochs_per_iter=1,
    batch_size=128,
    lr=1e-4,
    temperature=1.0,         # AZ: opening exploration; drops to ~0 after az_temperature_drop_ply
    entropy_coef=0.0,        # AZ doesn't need an entropy bonus; MCTS provides exploration
    value_coef=0.5,
    hidden_dim=ARCH["hidden_dim"],
    num_layers=ARCH["num_layers"],
    num_heads=ARCH["num_heads"],
    eval_every=5,
    eval_games=8,
    device=DEVICE,
    algo="az",
    mcts_sims=64,
    mcts_batch_size=8,
    az_temperature_drop_ply=30,
)
# train_rl(
#     sl_ckpt=SL_CKPT,
#     ckpt_dir=RL_CKPT_DIR,
#     iterations=50,
#     games_per_iter=8,
#     epochs_per_iter=2,
#     batch_size=128,
#     lr=1e-4,
#     temperature=0.7,
#     entropy_coef=0.01,
#     value_coef=0.5,
#     hidden_dim=ARCH["hidden_dim"],
#     num_layers=ARCH["num_layers"],
#     num_heads=ARCH["num_heads"],
#     eval_every=5,
#     eval_games=8,
#     device=DEVICE,
#     algo="ppo",
# )

OutOfMemoryError: CUDA out of memory. Tried to allocate 252.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 111.31 MiB is free. Process 3147 has 16.08 GiB memory in use. Process 23798 has 10.32 GiB memory in use. Process 27519 has 554.00 MiB memory in use. Process 36061 has 8.31 GiB memory in use. Process 78664 has 548.00 MiB memory in use. Including non-PyTorch memory, this process has 3.55 GiB memory in use. Of the allocated memory 2.85 GiB is allocated by PyTorch, and 213.13 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# --- load the fine-tuned model and compare head-to-head vs the SL baseline
from chess_gnn.train_rl import evaluate

latest = sorted(RL_CKPT_DIR.glob("rl_*.pt"))[-1]
print("loading", latest)

tuned = load_model(latest, device=DEVICE, **ARCH)
baseline = load_model(SL_CKPT, device=DEVICE, **ARCH)

stats = evaluate(tuned, baseline, device=DEVICE, num_games=20, temperature=0.05)
print("RL vs SL:", stats)

In [ ]:
# --- visualize predictions of the fine-tuned model -----------------------
import chess
from IPython.display import SVG, display

from chess_gnn.play import GNNAgent
from chess_gnn.viz import render_prediction_svg

# num_simulations=200 enables PUCT MCTS; set to 0 for raw policy (much faster).
# mcts_batch_size batches pending leaves during search, which helps most on GPU.
agent = GNNAgent(
    tuned,
    device=DEVICE,
    default_temperature=0.3,
    num_simulations=200,
    mcts_batch_size=8,
)

board = chess.Board()
board.push_san("e4"); board.push_san("c5"); board.push_san("Nf3")
ranking = agent.rank_moves(board)
for m, p in list(zip(ranking.moves, ranking.probabilities))[:8]:
    print(f"  {m.uci():>5s}  {p:.3f}")
display(SVG(render_prediction_svg(board, ranking, topk=6)))